# Notebook 05: DistMult and ComplEx Training

Training DistMult and ComplEx knowledge graph embedding models from scratch on DRKG using PyTorch. These serve as Baselines 2 and 3 in the benchmarking pipeline, alongside the pretrained TransE embeddings scored in Notebook 02.

Both models learn embeddings for all entities and relations in DRKG by optimizing a link prediction objective. The key difference:
- **DistMult** uses a bilinear diagonal scoring function — simple and effective but can only model symmetric relations
- **ComplEx** extends DistMult to complex-valued embeddings, handling both symmetric and antisymmetric relations

## Setup

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import csv
import os
import sys
sys.path.insert(1, '../utils')
from utils import download_and_extract

download_and_extract()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## Load and index the graph

Build entity and relation index maps from the training split.

In [2]:
# load train split
train_triples = []
with open('../train/drkg_train.tsv') as f:
    for line in f:
        h, r, t = line.strip().split('\t')
        train_triples.append((h, r, t))

# build entity and relation maps
entities = set()
relations = set()
for h, r, t in train_triples:
    entities.add(h)
    entities.add(t)
    relations.add(r)

entity2id = {e: i for i, e in enumerate(sorted(entities))}
relation2id = {r: i for i, r in enumerate(sorted(relations))}
id2entity = {i: e for e, i in entity2id.items()}

num_entities = len(entity2id)
num_relations = len(relation2id)

print(f"Entities: {num_entities:,}")
print(f"Relations: {num_relations:,}")
print(f"Train triples: {len(train_triples):,}")

Entities: 95,354
Relations: 107
Train triples: 5,286,834


## Dataset

In [3]:
class KGDataset(Dataset):
    def __init__(self, triples, entity2id, relation2id):
        self.triples = [
            (entity2id[h], relation2id[r], entity2id[t])
            for h, r, t in triples
            if h in entity2id and r in relation2id and t in entity2id
        ]
    
    def __len__(self):
        return len(self.triples)
    
    def __getitem__(self, idx):
        return self.triples[idx]

train_dataset = KGDataset(train_triples, entity2id, relation2id)
train_loader = DataLoader(train_dataset, batch_size=2048, shuffle=True)
print(f"Training batches: {len(train_loader)}")

Training batches: 2582


## DistMult Model

Scoring function: $f(h, r, t) = \langle \mathbf{e}_h, \mathbf{w}_r, \mathbf{e}_t \rangle$ — element-wise product of head, relation, and tail embeddings summed together.

In [4]:
class DistMult(nn.Module):
    def __init__(self, num_entities, num_relations, embedding_dim=400):
        super().__init__()
        self.entity_emb = nn.Embedding(num_entities, embedding_dim)
        self.relation_emb = nn.Embedding(num_relations, embedding_dim)
        nn.init.xavier_uniform_(self.entity_emb.weight)
        nn.init.xavier_uniform_(self.relation_emb.weight)
    
    def score(self, h, r, t):
        return (self.entity_emb(h) * self.relation_emb(r) * self.entity_emb(t)).sum(dim=-1)
    
    def forward(self, h, r, t, neg_t):
        pos_score = self.score(h, r, t)
        # score against random negative tails
        neg_score = self.score(h.unsqueeze(1).expand_as(neg_t), 
                               r.unsqueeze(1).expand_as(neg_t), neg_t)
        return pos_score, neg_score

## ComplEx Model

Extends DistMult to complex-valued embeddings, allowing the model to handle antisymmetric relations. The real and imaginary parts are stored as separate embedding layers.

In [5]:
class ComplEx(nn.Module):
    def __init__(self, num_entities, num_relations, embedding_dim=200):
        super().__init__()
        # real and imaginary parts
        self.entity_re = nn.Embedding(num_entities, embedding_dim)
        self.entity_im = nn.Embedding(num_entities, embedding_dim)
        self.relation_re = nn.Embedding(num_relations, embedding_dim)
        self.relation_im = nn.Embedding(num_relations, embedding_dim)
        for emb in [self.entity_re, self.entity_im, self.relation_re, self.relation_im]:
            nn.init.xavier_uniform_(emb.weight)
    
    def score(self, h, r, t):
        h_re = self.entity_re(h)
        h_im = self.entity_im(h)
        r_re = self.relation_re(r)
        r_im = self.relation_im(r)
        t_re = self.entity_re(t)
        t_im = self.entity_im(t)
        # hermitian dot product
        return (h_re * r_re * t_re 
                + h_re * r_im * t_im 
                + h_im * r_re * t_im 
                - h_im * r_im * t_re).sum(dim=-1)
    
    def forward(self, h, r, t, neg_t):
        pos_score = self.score(h, r, t)
        neg_score = self.score(h.unsqueeze(1).expand_as(neg_t),
                               r.unsqueeze(1).expand_as(neg_t), neg_t)
        return pos_score, neg_score

## Training function

Uses self-adversarial negative sampling following the approach in the original DistMult/ComplEx papers.

In [18]:
def train_model(model, train_loader, num_entities, epochs=5, lr=0.01, neg_samples=64, margin=1.0):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)
    model.train()
    
    for epoch in range(epochs):
        total_loss = 0
        for batch_idx, batch in enumerate(train_loader):
            h, r, t = [x.to(device) for x in batch]
            neg_t = torch.randint(0, num_entities, (h.size(0), neg_samples)).to(device)
            
            pos_score, neg_score = model(h, r, t, neg_t)
            
            # margin ranking loss
            pos_score = pos_score.unsqueeze(1).expand_as(neg_score)
            loss = F.relu(margin - pos_score + neg_score).mean()
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
            if batch_idx % 500 == 0:
                print(f"Epoch {epoch+1}, Batch {batch_idx}/{len(train_loader)}, Loss: {loss.item():.4f}")
        
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1} complete. Avg loss: {avg_loss:.4f}")
        scheduler.step()
    
    return model

## Train DistMult

In [19]:

distmult = DistMult(num_entities, num_relations, embedding_dim=400).to(device)
print(f"DistMult parameters: {sum(p.numel() for p in distmult.parameters()):,}")
distmult = train_model(distmult, train_loader, num_entities, epochs=5, lr=0.01, margin=1.0)

DistMult parameters: 38,184,400
Epoch 1, Batch 0/2582, Loss: 1.0000
Epoch 1, Batch 500/2582, Loss: 0.0580
Epoch 1, Batch 1000/2582, Loss: 0.0454
Epoch 1, Batch 1500/2582, Loss: 0.0399
Epoch 1, Batch 2000/2582, Loss: 0.0401
Epoch 1, Batch 2500/2582, Loss: 0.0367
Epoch 1 complete. Avg loss: 0.0797
Epoch 2, Batch 0/2582, Loss: 0.0232
Epoch 2, Batch 500/2582, Loss: 0.0233
Epoch 2, Batch 1000/2582, Loss: 0.0229
Epoch 2, Batch 1500/2582, Loss: 0.0256
Epoch 2, Batch 2000/2582, Loss: 0.0261
Epoch 2, Batch 2500/2582, Loss: 0.0238
Epoch 2 complete. Avg loss: 0.0252
Epoch 3, Batch 0/2582, Loss: 0.0217
Epoch 3, Batch 500/2582, Loss: 0.0177
Epoch 3, Batch 1000/2582, Loss: 0.0199
Epoch 3, Batch 1500/2582, Loss: 0.0172
Epoch 3, Batch 2000/2582, Loss: 0.0183
Epoch 3, Batch 2500/2582, Loss: 0.0202
Epoch 3 complete. Avg loss: 0.0192
Epoch 4, Batch 0/2582, Loss: 0.0162
Epoch 4, Batch 500/2582, Loss: 0.0178
Epoch 4, Batch 1000/2582, Loss: 0.0196
Epoch 4, Batch 1500/2582, Loss: 0.0188
Epoch 4, Batch 2000/2

## Score brain injury candidates with DistMult

In [20]:
brain_injury_disease_list = [
    'Disease::MESH:D020521',
    'Disease::MESH:D002544',
    'Disease::MESH:D020300',
    'Disease::MESH:D020520',
    'Disease::MESH:D002538',
    'Disease::MESH:D001930',
    'Disease::MESH:D006470',
]

treatment_relations = [
    'Hetionet::CtD::Compound:Disease',
    'GNBR::T::Compound:Disease'
]

# load drug list
drug_list = []
with open("../drug_repurpose/infer_drug.tsv", newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f, delimiter='\t', fieldnames=['drug', 'ids'])
    for row in reader:
        drug_list.append(row['drug'])

# map to ids
drug_ids = [entity2id[d] for d in drug_list if d in entity2id]
disease_ids = [entity2id[d] for d in brain_injury_disease_list if d in entity2id]
treatment_rids = [relation2id[r] for r in treatment_relations if r in relation2id]

print(f"Drugs mapped: {len(drug_ids)}")
print(f"Diseases mapped: {len(disease_ids)}")

Drugs mapped: 7939
Diseases mapped: 7


In [21]:
distmult.eval()
all_scores = []
all_drug_ids = []

with torch.no_grad():
    drug_tensor = torch.tensor(drug_ids).to(device)
    
    for rid in treatment_rids:
        r_tensor = torch.tensor([rid]).expand(len(drug_ids)).to(device)
        for did in disease_ids:
            t_tensor = torch.tensor([did]).expand(len(drug_ids)).to(device)
            scores = distmult.score(drug_tensor, r_tensor, t_tensor)
            all_scores.append(scores.cpu())
            all_drug_ids.append(drug_tensor.cpu())

all_scores = torch.cat(all_scores)
all_drug_ids = torch.cat(all_drug_ids)

idx = torch.flip(torch.argsort(all_scores), dims=[0])
all_scores = all_scores[idx].numpy()
all_drug_ids = all_drug_ids[idx].numpy()

_, unique_indices = np.unique(all_drug_ids, return_index=True)
topk_indices = np.sort(unique_indices)[:100]

distmult_results = pd.DataFrame({
    'rank': range(1, 101),
    'drug': [id2entity[int(d)] for d in all_drug_ids[topk_indices]],
    'score': all_scores[topk_indices]
})

distmult_results.head(20)

,rank,drug,score
0,1,Compound::DB00073,13.333157
1,2,Compound::DB00006,13.149067
2,3,Compound::DB00361,13.081741
3,4,Compound::DB00785,12.895241
4,5,Compound::DB00546,12.867630
5,6,Compound::DB00297,12.516942
6,7,Compound::DB00970,12.497654
7,8,Compound::DB00039,12.446999
8,9,Compound::DB00412,12.435107
9,10,Compound::DB00402,12.404100


## Save DistMult results and embeddings

In [22]:
os.makedirs('../results', exist_ok=True)
distmult_results.to_csv('../results/distmult_top100.csv', index=False)

# save embeddings for future use
torch.save(distmult.state_dict(), '../results/distmult_model.pt')
print("Saved.")

Saved.


## Train ComplEx

In [23]:
complex_model = ComplEx(num_entities, num_relations, embedding_dim=200).to(device)
print(f"ComplEx parameters: {sum(p.numel() for p in complex_model.parameters()):,}")
complex_model = train_model(complex_model, train_loader, num_entities, epochs=5)

ComplEx parameters: 38,184,400
Epoch 1, Batch 0/2582, Loss: 1.0000
Epoch 1, Batch 500/2582, Loss: 0.0600
Epoch 1, Batch 1000/2582, Loss: 0.0450
Epoch 1, Batch 1500/2582, Loss: 0.0415
Epoch 1, Batch 2000/2582, Loss: 0.0388
Epoch 1, Batch 2500/2582, Loss: 0.0323
Epoch 1 complete. Avg loss: 0.0814
Epoch 2, Batch 0/2582, Loss: 0.0239
Epoch 2, Batch 500/2582, Loss: 0.0256
Epoch 2, Batch 1000/2582, Loss: 0.0259
Epoch 2, Batch 1500/2582, Loss: 0.0265
Epoch 2, Batch 2000/2582, Loss: 0.0276
Epoch 2, Batch 2500/2582, Loss: 0.0251
Epoch 2 complete. Avg loss: 0.0257
Epoch 3, Batch 0/2582, Loss: 0.0205
Epoch 3, Batch 500/2582, Loss: 0.0184
Epoch 3, Batch 1000/2582, Loss: 0.0184
Epoch 3, Batch 1500/2582, Loss: 0.0190
Epoch 3, Batch 2000/2582, Loss: 0.0203
Epoch 3, Batch 2500/2582, Loss: 0.0191
Epoch 3 complete. Avg loss: 0.0196
Epoch 4, Batch 0/2582, Loss: 0.0166
Epoch 4, Batch 500/2582, Loss: 0.0161
Epoch 4, Batch 1000/2582, Loss: 0.0180
Epoch 4, Batch 1500/2582, Loss: 0.0175
Epoch 4, Batch 2000/25

## Score brain injury candidates with ComplEx

In [24]:
complex_model.eval()
all_scores_cx = []
all_drug_ids_cx = []

with torch.no_grad():
    drug_tensor = torch.tensor(drug_ids).to(device)
    
    for rid in treatment_rids:
        r_tensor = torch.tensor([rid]).expand(len(drug_ids)).to(device)
        for did in disease_ids:
            t_tensor = torch.tensor([did]).expand(len(drug_ids)).to(device)
            scores = complex_model.score(drug_tensor, r_tensor, t_tensor)
            all_scores_cx.append(scores.cpu())
            all_drug_ids_cx.append(drug_tensor.cpu())

all_scores_cx = torch.cat(all_scores_cx)
all_drug_ids_cx = torch.cat(all_drug_ids_cx)

idx = torch.flip(torch.argsort(all_scores_cx), dims=[0])
all_scores_cx = all_scores_cx[idx].numpy()
all_drug_ids_cx = all_drug_ids_cx[idx].numpy()

_, unique_indices = np.unique(all_drug_ids_cx, return_index=True)
topk_indices = np.sort(unique_indices)[:100]

complex_results = pd.DataFrame({
    'rank': range(1, 101),
    'drug': [id2entity[int(d)] for d in all_drug_ids_cx[topk_indices]],
    'score': all_scores_cx[topk_indices]
})

complex_results.head(20)

,rank,drug,score
0,1,Compound::DB00446,18.122448
1,2,Compound::DB09235,17.140938
2,3,Compound::DB12518,16.258865
3,4,Compound::DB09042,16.153694
4,5,Compound::DB01590,15.873960
5,6,Compound::DB01591,15.804358
6,7,Compound::DB00912,15.757977
7,8,Compound::DB01144,15.744793
8,9,Compound::DB00374,15.710096
9,10,Compound::DB13955,15.608527


## Save ComplEx results

In [25]:
complex_results.to_csv('../results/complex_top100.csv', index=False)
torch.save(complex_model.state_dict(), '../results/complex_model.pt')
print("Saved.")

Saved.


## Compare top candidates across baselines

Which drugs appear in the top 20 of both DistMult and ComplEx?

In [26]:
dm_top20 = set(distmult_results.head(20)['drug'])
cx_top20 = set(complex_results.head(20)['drug'])
overlap = dm_top20.intersection(cx_top20)

print(f"Drugs in top 20 of both DistMult and ComplEx: {len(overlap)}")
for d in overlap:
    print(f"  {d}")

Drugs in top 20 of both DistMult and ComplEx: 0


In [27]:
import pubchempy as pcp

def get_drug_name(drugbank_id):
    try:
        results = pcp.get_compounds(drugbank_id, 'name')
        if results:
            syns = results[0].synonyms
            return syns[0] if syns else 'unknown'
        return 'unknown'
    except:
        return 'unknown'

# look up DistMult top 20
distmult_results['drug_name'] = distmult_results['drug'].apply(
    lambda x: get_drug_name(x.replace('Compound::', ''))
)

print("DistMult top 20:")
print(distmult_results[['rank', 'drug_name', 'drug', 'score']].head(20).to_string(index=False))

DistMult top 20:
 rank                  drug_name              drug     score
    1                    unknown Compound::DB00073 13.333157
    2                Bivalirudin Compound::DB00006 13.149067
    3                vinorelbine Compound::DB00361 13.081741
    4                    unknown Compound::DB00785 12.895241
    5                 ADINAZOLAM Compound::DB00546 12.867630
    6 Bupivacaine [USAN:INN:BAN] Compound::DB00297 12.516942
    7               DACTINOMYCIN Compound::DB00970 12.497654
    8                    unknown Compound::DB00039 12.446999
    9              rosiglitazone Compound::DB00412 12.435107
   10                    unknown Compound::DB00402 12.404100
   11                clofazimine Compound::DB00845 12.347299
   12               dipyridamole Compound::DB00975 12.345018
   13                    unknown Compound::DB00040 12.327597
   14               cerivastatin Compound::DB00439 12.255051
   15                  goserelin Compound::DB00014 12.132320
   16  

In [28]:
complex_results['drug_name'] = complex_results['drug'].apply(
    lambda x: get_drug_name(x.replace('Compound::', ''))
)

print("ComplEx top 20:")
print(complex_results[['rank', 'drug_name', 'drug', 'score']].head(20).to_string(index=False))

ComplEx top 20:
 rank             drug_name              drug     score
    1       chloramphenicol Compound::DB00446 18.122448
    2           Efonidipine Compound::DB09235 17.140938
    3            RACLOPRIDE Compound::DB12518 16.258865
    4   Tedizolid phosphate Compound::DB09042 16.153694
    5            Everolimus Compound::DB01590 15.873960
    6           Solifenacin Compound::DB01591 15.804358
    7           Repaglinide Compound::DB00912 15.757977
    8               unknown Compound::DB01144 15.744793
    9               unknown Compound::DB00374 15.710096
   10 ESTRADIOL DIENANTHATE Compound::DB13955 15.608527
   11          benzthiazide Compound::DB00562 15.499275
   12           sumatriptan Compound::DB00669 15.418424
   13            Pranlukast Compound::DB01411 15.244236
   14            OLODATEROL Compound::DB09080 15.221987
   15             alfuzosin Compound::DB00346 15.071980
   16            Vismodegib Compound::DB08828 15.006698
   17           nateglinide Comp

In [ ]:
distmult_results.to_csv('../results/distmult_top20_named.csv', index=False)
complex_results.to_csv('../results/complex_top20_named.csv', index=False)

: 

## Summary

- Trained DistMult and ComplEx from scratch on DRKG (5.28M training triples)
- Scored 8,104 candidate drugs against 7 acute brain injury disease nodes
- Results saved to `results/distmult_top100.csv` and `results/complex_top100.csv`
- Next: Notebook 06 — GraphSAGE training